# 03 - Absolute Sustainability Ratio

    ASR = emissions / allocated carrying capacity

The numerator is notebook 02's emissions table. The denominator is each
country's fair share of the **2 °C steady-state carrying capacity** - a fixed
annual level of greenhouse-gas emissions the climate could sustain
indefinitely. Three allocation rules divide it, and
the piece compares them: **equal per capita**, where every person alive gets the
same share; **prioritarian**, which weights the share towards poorer countries;
and **grandfathering**, which hands each country the share it already had in a
reference year.

**Static, not the dynamic AR6 pathway.** The dynamic budget is designed for
prospective work: it front-loads the allowance and puts the reductions after
this study's window. Measured against it the world sits at 1.17x its 2023
allowance, which reads as near-compliant and tells you nothing. Against the
static carrying capacity the world emits **6.4x** what it may. For a historical
snapshot the static value is the one that means something.

The carrying capacity comes from pyaesa's shipped table (Bjorn & Hauschild
2015; Sala et al. 2020; Sanye-Mengual & Sala 2023) and is bracketed:

| bound | kg CO2-eq/yr | level | per person, 2023 |
|---|---|---|---|
| `min_cc` | 6.81e12 | 2 °C steady state - the headline | 0.847 t |
| `max_cc` | 8.72e12 | 128% of `min_cc` (the 450/350 ppm ratio) | 1.084 t |

- **ASR < 1** - living inside its allocation
- **ASR > 1** - exceeding it, by that multiple

**Outputs**
- `data_viz/asr.json` - `{iso3: {year: asr}}`, equal per capita at `min_cc`
- `data_viz/asr_gdp.json` - the same under prioritarian allocation
- `data_viz/asr_gf.json` - the same under grandfathering
- `data_viz/asr.csv` - long form, all three allocation rules and both bounds

In [ ]:
import json

import pandas as pd
import pyaesa

from config import (
    ACC_FILE, ASR_FILE, ASR_FILE_GDP, CC_BOUND, CC_FILE, FU_CODE, GF_BASE_YEAR,
    LCA_FILE, LCA_VERSION, LCIA_METHOD, PROJECT, REGION_COL, ROOT, VIZ, YEARS,
    YEAR_COLS,
)

pyaesa.set_workspace(top_path=str(ROOT))

## 1. Compute

`r_p` is pinned to the countries actually present in the emissions table.
Without it pyaesa allocates a budget to every World Bank country and then
fails on the ones with no emissions data to divide.

`base_cc_args` turns the dynamic AR6 path off and the static one on. That is
the whole switch - and it removes four choices the dynamic path forced with no
principled way to settle them: which IAM, which SSP, which climate pathway, and
which of the twelve C1 scenarios to report. pyaesa offers `uncertainty_asr` to
propagate those by Monte Carlo; picking one representative, as this notebook
used to, was not a valid substitute. Static has no such axes: the uncertainty is
the `min_cc`/`max_cc` bracket, and both are computed.

`figures=False` keeps the run to a few minutes. pyaesa would otherwise render
about two thousand PNGs that nothing here reads.

pyaesa runs the whole chain - allocation, then carrying capacity, then the
ratio - so there is no separate step to call first.

### If pyaesa refuses to run

pyaesa fingerprints every computed phase by the exact country set -
`r_p-n198_ABW`. Change which countries notebook 02 emits and the stored
fingerprint no longer matches, so it stops rather than mixing two vintages in
one folder:

```
ValueError: ... studied_indices_tag: requested='r_p-n198_ABW';
                                     persisted='r_p-n206_ABW'
```

`refresh=True` does **not** fix this. It validates branch identity *before* it
cleans, so it fails on the same mismatch one phase later:

```
ValueError: deterministic_acc cannot append to deterministic scope
            'iso3_static_gwp100_lcia' because branch identity changed.
```

The fix is to clear the computed phases and let them rebuild. Everything under
`asr/` is derived except `A_lca`, which is notebook 02's output and must be
kept:

```
mkdir -p asr/_stale && mv asr/B1_asocc/iso3 asr/B2_acc asr/C_asr asr/_stale/
```

Then run this notebook with `refresh=False`, as it is. Delete `asr/_stale`
once the run succeeds.

In [ ]:
countries = sorted(pd.read_csv(LCA_FILE)[REGION_COL].unique())
print(f"Computing ASR for {len(countries)} countries...")

result = pyaesa.deterministic_asr(
    project_name=PROJECT,
    source="iso3",
    fu_code=FU_CODE,
    years=list(YEARS),
    lcia_method=LCIA_METHOD,
    r_p=countries,
    base_asocc_args={"include_lcia_based_allocation_methods": False},
    base_cc_args={
        "static": {"active": True, "exclude_max_cc": False},
        "dynamic_ar6": {"active": False},
    },
    lca_args={"external_lca": {"active": True, "version_name": LCA_VERSION}},
    figures=False,
)

## 2. Inspect

The static output carries one row per country per `cc_bound`. `min_cc` is the
2 °C steady-state budget and the headline; `max_cc`, 128% of it (the 450/350
ppm ratio), gives the generous end of the bracket.

Palau and New Caledonia sit far above their neighbours. Both are territorial
emissions divided by a small resident population - Palau hosts several times
its own population in visitors each year, New Caledonia runs nickel smelters.
Worth flagging in the visualisation rather than presenting flat.

In [ ]:
cc = pd.read_csv(CC_FILE).iloc[0]
LATEST = max(YEARS)
pop_world = pd.read_csv(VIZ / "emissions.csv").query("year == @LATEST")["population"].sum()
print(f"fair share per person, {LATEST}:  "
      f"min_cc {cc.min_cc / pop_world / 1000:.3f} t   "
      f"max_cc {cc.max_cc / pop_world / 1000:.3f} t")

frames = []
for path, method in [(ASR_FILE, "equal_per_capita"), (ASR_FILE_GDP, "prioritarian_gdp")]:
    df = pd.read_csv(path)
    frames.append(
        df.melt(id_vars=[REGION_COL, "cc_bound"], value_vars=YEAR_COLS,
                var_name="year", value_name="asr")
          .astype({"year": int})
          .rename(columns={REGION_COL: "iso_code"})
          .assign(method=method)
    )
long = pd.concat(frames, ignore_index=True)

snapshot = (
    long.query("method == 'equal_per_capita' and cc_bound == @CC_BOUND and year == @LATEST")
        .set_index("iso_code")["asr"].sort_values()
)
print(f"\n{len(snapshot)} countries, {LATEST}, equal per capita at {CC_BOUND}\n")
print("Lowest 10:\n", snapshot.head(10).round(3), "\n")
print("Highest 10:\n", snapshot.tail(10).round(1))


## 3. Grandfathering

pyaesa will not compute this one. It knows the rule as **AR(E)** - acquired
rights - and defines it exactly as you would expect: each country's share of
world emissions in one fixed reference year, held constant thereafter. But
`source="iso3"` is gated to `EG(Pop)` and `PR(GDPcap)` and raises on anything
else, because the other six L1 methods in its registry all resolve their shares
from LCIA-weighted MRIO impacts, which a country-level emissions table does not
carry.

The equation itself needs only emissions, so it is applied here directly, on
the same inputs and against the same budget:

    share(c) = E(c, base) / sum_c E(c, base)

The denominator is **pyaesa's own allocated carrying capacity**, summed back
over the 198 countries rather than taken raw from the shipped table. That is
99.8% of the global figure - the missing 0.2% went to countries with no
emissions data - and using it means all three rules divide an identical pool.
The only thing that changes between them is who gets what.

**The reference year is load-bearing.** Set it to the current year and the rule
collapses: `share(c, t) = E(c, t) / E_world(t)` cancels the numerator, every
country lands on the same ratio, and the map goes flat at the world figure.
That degeneracy *is* the argument against grandfathering - a rule under which
nobody is ever a worse offender than anybody else - but it does not draw. The
window start is used instead, which is also the only anchor the emissions table
offers.

So read the grandfathering numbers as **divergence since 2000**, not as a level:
every country starts the window on the identical ratio by construction, and
moves off it only by growing faster or slower than the world.

In [ ]:
emissions = pd.read_csv(VIZ / "emissions.csv")

base = emissions.query("year == @GF_BASE_YEAR").set_index("iso_code")["emissions_kg"]
gf_share = base / base.sum()

pool = (pd.read_csv(ACC_FILE)
          .melt(id_vars="cc_bound", value_vars=YEAR_COLS, var_name="year", value_name="acc")
          .astype({"year": int})
          .groupby(["cc_bound", "year"], as_index=False)["acc"].sum())

gf = (emissions[["iso_code", "year", "emissions_kg"]]
        .merge(pool, on="year")
        .assign(asr=lambda d: d.emissions_kg / (d.acc * d.iso_code.map(gf_share)),
                method="grandfathering")
      [["iso_code", "cc_bound", "year", "asr", "method"]])
long = pd.concat([long, gf], ignore_index=True)

flat = gf.query("cc_bound == @CC_BOUND and year == @GF_BASE_YEAR")["asr"]
print(f"{GF_BASE_YEAR} is flat by construction: "
      f"{flat.min():.4f}-{flat.max():.4f} across {len(flat)} countries "
      f"(= world emissions / budget that year)\n")

gf_now = (gf.query("cc_bound == @CC_BOUND and year == @LATEST")
            .set_index("iso_code")["asr"].sort_values())
print(f"{LATEST}, grandfathering at {CC_BOUND}\n")
print("Lowest 10 (shrunk most since the base year):\n", gf_now.head(10).round(3), "\n")
print("Highest 10 (grew most):\n", gf_now.tail(10).round(1))

## 4. Export

In [ ]:
VIZ.mkdir(exist_ok=True)
long.to_csv(VIZ / "asr.csv", index=False)

for method, fname in [("equal_per_capita", "asr.json"),
                      ("prioritarian_gdp", "asr_gdp.json"),
                      ("grandfathering", "asr_gf.json")]:
    sub = long.query("method == @method and cc_bound == @CC_BOUND")
    export = {iso: dict(zip(g["year"], g["asr"].round(4))) for iso, g in sub.groupby("iso_code")}
    (VIZ / fname).write_text(json.dumps(export, indent=2))
    print(f"{len(export):>4} countries -> data_viz/{fname}")

print(f"{len(long):>4} rows      -> data_viz/asr.csv  (all three methods, both bounds)")

## 5. The underlying variables

One table with every input and intermediate behind the three ratios, so a
visualisation can show any stage of the chain without recomputing it.

Three variables per country per year - emissions, population, GDP - plus two
global constants. Everything else (per-capita figures, allocation shares, the
allocated carrying capacity, all three ASRs) is derived from those five.


In [ ]:
em = pd.read_csv(VIZ / "emissions.csv")
wb = pd.read_csv(ROOT / "data_processed" / "pop_gdp" / "wb_processed.csv")

def wb_long(variable, name):
    return (wb.query("variable == @variable")
              .melt(id_vars="iso3_code", value_vars=YEAR_COLS, var_name="year", value_name=name)
              .astype({"year": int}).rename(columns={"iso3_code": "iso_code"}))

gdp = wb_long("GDP|PPP", "gdp_ppp_2017usd")
wide = (long.query("cc_bound == @CC_BOUND")
            .pivot_table(index=["iso_code", "year"], columns="method", values="asr")
            .rename(columns={"equal_per_capita": "asr_eg", "prioritarian_gdp": "asr_pr",
                             "grandfathering": "asr_gf"})
            .reset_index())

v = (em.merge(gdp, on=["iso_code", "year"], how="left")
       .merge(wide, on=["iso_code", "year"], how="left")
       .assign(gdp_per_capita=lambda d: d.gdp_ppp_2017usd / d.population,
               fair_share_t=lambda d: d.emissions_t_per_capita / d.asr_eg))
v.to_csv(VIZ / "variables.csv", index=False)

print(f"{len(v):,} country-years -> data_viz/variables.csv")
print(f"fair share {CC_BOUND}: {v.fair_share_t.min():.3f}–{v.fair_share_t.max():.3f} t "
      f"(flat across countries, falls as population grows)")
